In [1]:
import pandas as pd
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity

train = pd.read_csv("train.csv")

In [2]:
def clean(text):
    text = str(text).lower()

    for p in string.punctuation:
        text = text.replace(p, "")

    return text

In [3]:
counts = train["answer"].value_counts()
print(counts)

ans = counts.max() + counts.min()
print(ans)

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
814


In [4]:
words = []

for prompt in train["prompt"]:
    prompt = clean(prompt)
    parts = prompt.split()

    for word in parts:
        words.append(word)

unique_words = set(words)

print(len(unique_words))

859


In [5]:
row = train[train["id"] == 1].iloc[0]

prompt = clean(row["prompt"])
words = prompt.split()

filtered = []

for word in words:
    if word not in ENGLISH_STOP_WORDS:
        filtered.append(word)

print(filtered)
print(len(filtered))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
13


In [6]:
corpus = []

for i in range(len(train)):
    text = (
        str(train.loc[i, "prompt"]) + " " +
        str(train.loc[i, "A"]) + " " +
        str(train.loc[i, "B"]) + " " +
        str(train.loc[i, "C"]) + " " +
        str(train.loc[i, "D"]) + " " +
        str(train.loc[i, "E"])
    )

    corpus.append(clean(text))

In [7]:
tfidf = TfidfVectorizer(stop_words="english")
X = tfidf.fit_transform(corpus)

print(X.shape[1])

2865


In [8]:
row = train[train["id"] == 1].iloc[0]

prompt = clean(row["prompt"])
option = clean(row["A"])

vectors = tfidf.transform([prompt, option])

score = cosine_similarity(vectors[0], vectors[1])[0][0]

print(round(score, 4))

0.1866


In [9]:
labels = ["A", "B", "C", "D", "E"]
correct = 0
total = len(train)

for i in range(total):
    prompt = clean(train.loc[i, "prompt"])

    scores = []

    for label in labels:
        option = clean(train.loc[i, label])

        vectors = tfidf.transform([prompt, option])
        score = cosine_similarity(vectors[0], vectors[1])[0][0]

        scores.append(score)

    max_index = scores.index(max(scores))
    prediction = labels[max_index]

    if prediction == train.loc[i, "answer"]:
        correct += 1

accuracy = (correct / total) * 100
print(accuracy)

11.3


In [10]:
def score(actual, pred):
    if actual in pred:
        rank = pred.index(actual) + 1
        return 1 / rank
    return 0

In [11]:
counts = train["answer"].value_counts()

top3 = list(counts.index[:3])
print(top3)

['B', 'C', 'A']


In [12]:
scores = []

for ans in train["answer"]:
    scores.append(score(ans, top3))

print(sum(scores)/len(scores))

0.42125


In [13]:
labels = ["A", "B", "C", "D", "E"]
all_scores = []

for i in range(len(train)):
    prompt = clean(train.loc[i, "prompt"])

    sims = []

    for label in labels:
        option = clean(train.loc[i, label])

        vectors = tfidf.transform([prompt, option])
        sim = cosine_similarity(vectors[0], vectors[1])[0][0]

        sims.append((label, sim))

    sims.sort(key=lambda x: x[1], reverse=True)

    top3 = [sims[0][0], sims[1][0], sims[2][0]]

    actual = train.loc[i, "answer"]

    all_scores.append(score(actual, top3))

print(sum(all_scores)/len(all_scores))

0.2825833333333333
